**使用指南：**

！！！使用前，请按需修改下面的环境配置与映射

1. 首次使用，先 `Run all` 运行一次，弹出 widgets 输入框（`retry_type`）
2. 输入 `retry_type`（仅支持：`CBR`、`CBRDrjart`、`CBRPublic`）后回车
3. 脚本会：
   - 全量读取 `retry_config_downstream` 的 `MarketCode` + `MDMKey`
   - 先到对应 dataset 表取 JSON，未命中则降级到 snapshot delta table 回查
   - 根据 `MarketCode` 映射目标 topic 后重新发送 Kafka
   - 重发成功后仅删除已发送的配置行

In [0]:
## ===================== 环境配置 =======================
# Kafka broker 地址
kafka_brokers = "10.249.200.43:9092,10.249.200.42:9092,10.249.200.44:9092"
# database & table 配置
golden_consumer_combine_database = "catalog_southeastasia_mdm_golden_prod.consumer_combine"
# 重试配置表（全量读取此表中的 MarketCode + MDMKey）
retry_config_downstream_table = "catalog_southeastasia_mdm_share_prod.share_mdm_config.retry_config_downstream"
## ======================================================

# retry_type 配置（字典字段：dataset_table, json_column_candidates, topic_template）
RETRY_TYPE_CONFIG = {
    "CBR": {
        "dataset_table": f"{golden_consumer_combine_database}.t_cbr_dataset",
        "json_column_candidates": ["FinalJSON"],
        "topic_template": "CBR_{suffix}",
    },
    "CBRDrjart": {
        "dataset_table": f"{golden_consumer_combine_database}.t_cbrdrjart_dataset",
        "json_column_candidates": ["FinalJSON"],
        "topic_template": "CBR_DrJart_{suffix}",
        # CBRDrjart 仅处理韩国市场数据。
        "force_market_code": "KOR",
    },
    "CBRPublic": {
        "dataset_table": f"{golden_consumer_combine_database}.t_cbr_withoutpii_dataset",
        "json_column_candidates": ["FinalJSON"],
        "topic_template": "CBRPublic_{suffix}",
    },
}

# 市场缩写映射（用于拼装 topic 名称）
MARKET_TOPIC_SUFFIX = {
    # "AUS": "AU",
    "HKG": "HK",
    "IDN": "ID",
    "JPN": "JP",
    "KOR": "KR",
    "MYS": "MY",
    "NZL": "NZ",
    "PHL": "PH",
    "SGP": "SG",
    "THA": "TH",
    "TWN": "TW",
    "VNM": "VN",
}

# 若某些市场需要特殊 topic，可在此覆盖（key: (retry_type, market_code)）
RETRY_TYPE_MARKET_TOPIC_OVERRIDE = {
    # ("CBRDrjart", "KOR"): "CBR_DrJart_KR_TS",
    ("CBR", "AUS"): "ConsumerBestRecordTopic",
    ("CBRPublic", "AUS"): "CBRPublicTopic"
}
# (retry_type, market_code) → snapshot delta table path（用于 dataset 查不到时的降级回查）
SNAPSHOT_PATH_MAP = {
    ("CBR", "AUS"): "abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_prod/history_data_loading/Prod_Consumer_20260715/AUS/aus_elcconsumermdm/cbrl",
    ("CBR", "HKG"): "abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_prod/history_data_loading/Prod_Consumer_20260715/HKG/hkg_elcconsumermdm/cbrl",
    ("CBR", "IDN"): "abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_prod/history_data_loading/Prod_Consumer_20260715/IDN/idn_elcconsumermdm/cbrl",
    ("CBR", "JPN"): "abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_prod/history_data_loading/Prod_Consumer_20260715/JPN/jpn_elcconsumermdm/cbrl_inc",
    ("CBR", "KOR"): "abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_prod/history_data_loading/Prod_Consumer_20260715/KOR/kor_elcconsumermdm/cbrl",
    ("CBR", "MYS"): "abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_prod/history_data_loading/Prod_Consumer_20260715/MYS/mys_elcconsumermdm/cbrl",
    ("CBR", "NZL"): "abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_prod/history_data_loading/Prod_Consumer_20260715/NZL/nzl_elcconsumermdm/cbrl",
    ("CBR", "PHL"): "abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_prod/history_data_loading/Prod_Consumer_20260715/PHL/phl_elcconsumermdm/cbrl",
    ("CBR", "SGP"): "abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_prod/history_data_loading/Prod_Consumer_20260715/SGP/sgp_elcconsumermdm/cbrl",
    ("CBR", "THA"): "abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_prod/history_data_loading/Prod_Consumer_20260715/THA/tha_elcconsumermdm/cbrl",
    ("CBR", "TWN"): "abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_prod/history_data_loading/Prod_Consumer_20260715/TWN/twn_elcconsumermdm/cbrl",
    ("CBR", "VNM"): "abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_prod/history_data_loading/Prod_Consumer_20260715/VNM/vnm_elcconsumermdm/cbrl",
    ("CBRPublic", "AUS"): "abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_prod/history_data_loading/Prod_Consumer_20260715/AUS/aus_elcconsumermdm/cbrlpublic",
    ("CBRPublic", "HKG"): "abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_prod/history_data_loading/Prod_Consumer_20260715/HKG/hkg_elcconsumermdm/cbrlpublic",
    ("CBRPublic", "IDN"): "abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_prod/history_data_loading/Prod_Consumer_20260715/IDN/idn_elcconsumermdm/cbrlpublic",
    ("CBRPublic", "JPN"): "abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_prod/history_data_loading/Prod_Consumer_20260715/JPN/jpn_elcconsumermdm/cbrlpublic_inc",
    ("CBRPublic", "KOR"): "abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_prod/history_data_loading/Prod_Consumer_20260715/KOR/kor_elcconsumermdm/cbrlpublic",
    ("CBRPublic", "MYS"): "abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_prod/history_data_loading/Prod_Consumer_20260715/MYS/mys_elcconsumermdm/cbrlpublic",
    ("CBRPublic", "NZL"): "abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_prod/history_data_loading/Prod_Consumer_20260715/NZL/nzl_elcconsumermdm/cbrlpublic",
    ("CBRPublic", "PHL"): "abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_prod/history_data_loading/Prod_Consumer_20260715/PHL/phl_elcconsumermdm/cbrlpublic",
    ("CBRPublic", "SGP"): "abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_prod/history_data_loading/Prod_Consumer_20260715/SGP/sgp_elcconsumermdm/cbrlpublic",
    ("CBRPublic", "THA"): "abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_prod/history_data_loading/Prod_Consumer_20260715/THA/tha_elcconsumermdm/cbrlpublic",
    ("CBRPublic", "TWN"): "abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_prod/history_data_loading/Prod_Consumer_20260715/TWN/twn_elcconsumermdm/cbrlpublic",
    ("CBRPublic", "VNM"): "abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_prod/history_data_loading/Prod_Consumer_20260715/VNM/vnm_elcconsumermdm/cbrlpublic",
    ("CBRDrjart", "KOR"): "abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_prod/history_data_loading/Prod_Consumer_20260715/KOR/kor_elcconsumermdm/cbrldrjart",
}
## ======================================================

from pyspark.sql import functions as F

# Databricks widgets: 重试类型
# dbutils.widgets.dropdown("retry_type", "CBR", ["CBR", "CBRDrjart", "CBRPublic"])
# "CBR", "CBRDrjart", "CBRPublic"
retry_type = dbutils.widgets.get("retry_type")

print(f"retry_type: {retry_type}")
print(f"kafka brokers: {kafka_brokers}")
print(f"retry config table: {retry_config_downstream_table}")

def _find_column_name(df, expected_col_name):
    """
    在 DataFrame 中按不区分大小写查找列名，返回真实列名。

    目的：
    - 兼容配置表/数据表大小写差异（如 MarketCode vs marketcode）。
    - 失败时尽早抛错，避免后续 join 结果为空但不易排查。
    """
    target = expected_col_name.upper()
    for c in df.columns:
        if c.upper() == target:
            return c
    raise ValueError(f"Column [{expected_col_name}] not found in DataFrame columns: {df.columns}")

def _pick_json_column(dataset_df, json_column_candidates, retry_type):
    """
    按 retry_type 的候选配置，从 dataset 表中挑选 JSON 列。

    返回:
        str: dataset 表中实际存在的 JSON 列名。
    """
    candidates = json_column_candidates or []
    if len(candidates) == 0:
        raise ValueError(f"No json column candidates configured for retry_type: {retry_type}")

    dataset_cols_upper_map = {c.upper(): c for c in dataset_df.columns}
    for candidate in candidates:
        real_name = dataset_cols_upper_map.get(candidate.upper())
        if real_name is not None:
            return real_name

    raise ValueError(
        f"No matched json column for retry_type={retry_type}. "
        f"candidates={candidates}, dataset columns={dataset_df.columns}"
    )

def _build_topic_mapping_df(retry_type, market_codes_df):
    """
    根据 retry_type + MarketCode 生成 topic 映射 DataFrame（market_code, kafka_topic）。

    规则：
    1. 先查 RETRY_TYPE_MARKET_TOPIC_OVERRIDE；
    2. 否则用 RETRY_TYPE_CONFIG[retry_type]["topic_template"] + MARKET_TOPIC_SUFFIX 拼装。
    """
    retry_type_config = RETRY_TYPE_CONFIG.get(retry_type)
    if retry_type_config is None:
        raise ValueError(f"Unsupported retry_type: {retry_type}")

    template = retry_type_config.get("topic_template")
    if not template:
        raise ValueError(f"No topic template configured for retry_type: {retry_type}")

    topic_rows = []
    for row in market_codes_df.collect():
        market_code = str(row["market_code"]).strip().upper()

        override_topic = RETRY_TYPE_MARKET_TOPIC_OVERRIDE.get((retry_type, market_code))
        if override_topic:
            topic_rows.append((market_code, override_topic))
            continue

        # 简化规则：CBRDrjart 固定使用 KR 后缀。
        if retry_type == "CBRDrjart":
            suffix = "KR"
        else:
            suffix = MARKET_TOPIC_SUFFIX.get(market_code)

        if suffix is None:
            topic_rows.append((market_code, None))
            continue

        topic_rows.append((market_code, template.format(suffix=suffix)))

    return spark.createDataFrame(topic_rows, ["market_code", "kafka_topic"])


def _read_snapshot_for_misses(misses_df, retry_type, json_column_candidates):
    """
    对 dataset 中未命中的候选记录，尝试从 snapshot delta table 路径回查 JSON。

    参数:
        misses_df (DataFrame): 含 market_code + mdm_key，均为 dataset 未命中记录。
        retry_type (str)
        json_column_candidates (list[str])

    返回:
        (DataFrame, int): market_code, mdm_key, json_value 及总命中数。
                         若所有 market 都无 snapshot 路径或读取失败，返回 (空 DataFrame, 0)。
    """
    markets = [r["market_code"] for r in misses_df.select("market_code").distinct().collect()]
    snapshot_dfs = []
    total_hits = 0

    for market_code in markets:
        path = SNAPSHOT_PATH_MAP.get((retry_type, market_code))
        if path is None:
            print(f"[WARN] No snapshot path configured for retry_type={retry_type}, market={market_code}")
            continue

        try:
            raw_df = spark.read.format("delta").load(path)

            mdm_key_col = _find_column_name(raw_df, "MDMKey")
            json_col = _pick_json_column(raw_df, json_column_candidates, retry_type)

            market_miss_keys_df = (
                misses_df
                .filter(F.col("market_code") == market_code)
                .select("mdm_key").distinct()
            )

            snap_hits = (
                raw_df
                .select(
                    F.trim(F.col(mdm_key_col)).alias("mdm_key"),
                    F.col(json_col).cast("string").alias("json_value"),
                )
                .filter(F.col("mdm_key").isNotNull() & (F.length(F.col("mdm_key")) > 0))
                .join(market_miss_keys_df, "mdm_key", "inner")
                .withColumn("market_code", F.lit(market_code))
                .select("market_code", "mdm_key", "json_value")
            )

            hit_count = snap_hits.count()
            print(f"[INFO] Snapshot hits for {retry_type}/{market_code}: {hit_count}")
            total_hits += hit_count
            if hit_count > 0:
                snapshot_dfs.append(snap_hits)

        except Exception as e:
            print(f"[WARN] Failed to process snapshot for retry_type={retry_type}, market={market_code}, path={path}: {e}")
            continue

    if snapshot_dfs:
        from functools import reduce
        return reduce(lambda a, b: a.unionByName(b), snapshot_dfs), total_hits
    # ponytail: 空 DataFrame 带正确 schema，让调用方 unionByName 不炸
    return misses_df.select("market_code", "mdm_key").withColumn("json_value", F.lit("").cast("string")).limit(0), 0


def find_retry_candidate_df(retry_type):
    """
    全量读取 retry_config_downstream，并提取去重后的 MarketCode + MDMKey。

    返回:
        DataFrame: market_code, mdm_key
    """
    config_df = spark.table(retry_config_downstream_table)

    market_col = _find_column_name(config_df, "MarketCode")
    mdm_key_col = _find_column_name(config_df, "MDMKey")

    retry_type_config = RETRY_TYPE_CONFIG.get(retry_type, {})
    force_market_code = retry_type_config.get("force_market_code")

    candidate_df = (
        config_df
        .select(
            F.upper(F.trim(F.col(market_col))).alias("market_code"),
            F.trim(F.col(mdm_key_col)).alias("mdm_key"),
        )
        .filter(F.col("market_code").isNotNull() & (F.length(F.col("market_code")) > 0))
        .filter(F.col("mdm_key").isNotNull() & (F.length(F.col("mdm_key")) > 0))
        .dropDuplicates(["market_code", "mdm_key"])
    )

    # 配置了强制市场时，只处理指定 market_code 的候选记录。
    if force_market_code:
        candidate_df = candidate_df.filter(F.col("market_code") == F.lit(str(force_market_code).strip().upper()))
    return candidate_df

def build_kafka_retry_df(candidate_df, retry_type):
    """
    依据 retry_type 从对应 dataset 表取 JSON，未命中时降级到 snapshot delta table，
    生成 Kafka 发送数据。

    参数:
        candidate_df (DataFrame): market_code + mdm_key 候选集合。
        retry_type (str): CBR / CBRDrjart / CBRPublic

    返回:
        DataFrame: market_code, mdm_key, kafka_topic, kafka_key, kafka_value
    """
    retry_type_config = RETRY_TYPE_CONFIG.get(retry_type)
    if retry_type_config is None:
        raise ValueError(f"Unsupported retry_type: {retry_type}")

    dataset_table = retry_type_config.get("dataset_table")
    json_column_candidates = retry_type_config.get("json_column_candidates")
    if not dataset_table:
        raise ValueError(f"No dataset_table configured for retry_type: {retry_type}")

    # ---- 1. 从 dataset 表回查 JSON ----
    dataset_df = spark.table(dataset_table)

    dataset_market_col = _find_column_name(dataset_df, "MarketCode")
    dataset_mdm_key_col = _find_column_name(dataset_df, "MDMKey")
    dataset_json_col = _pick_json_column(dataset_df, json_column_candidates, retry_type)

    dataset_norm_df = (
        dataset_df
        .select(
            F.upper(F.trim(F.col(dataset_market_col))).alias("market_code"),
            F.trim(F.col(dataset_mdm_key_col)).alias("mdm_key"),
            F.col(dataset_json_col).cast("string").alias("json_value"),
        )
        .filter(F.col("market_code").isNotNull() & (F.length(F.col("market_code")) > 0))
        .filter(F.col("mdm_key").isNotNull() & (F.length(F.col("mdm_key")) > 0))
    )

    dataset_hits_df = (
        candidate_df.alias("c")
        .join(
            dataset_norm_df.alias("d"),
            (F.col("c.market_code") == F.col("d.market_code"))
            & (F.col("c.mdm_key") == F.col("d.mdm_key")),
            "inner",
        )
        .select("d.market_code", "d.mdm_key", "d.json_value")
    )

    dataset_hit_count = dataset_hits_df.count()
    print(f"[INFO] Dataset hits: {dataset_hit_count}")

    # ---- 2. 未命中记录降级到 snapshot delta table ----
    total_candidate_count = candidate_df.count()
    if dataset_hit_count < total_candidate_count:
        misses_df = (
            candidate_df.alias("c")
            .join(
                dataset_hits_df.select("market_code", "mdm_key").alias("h"),
                (F.col("c.market_code") == F.col("h.market_code"))
                & (F.col("c.mdm_key") == F.col("h.mdm_key")),
                "left_anti",
            )
        )
        snapshot_hits_df, snapshot_hit_count = _read_snapshot_for_misses(misses_df, retry_type, json_column_candidates)
        print(f"[INFO] Snapshot fallback hits: {snapshot_hit_count}")
    else:
        misses_df = None
        snapshot_hits_df = None
        snapshot_hit_count = 0

    # ---- 3. 合并 dataset + snapshot 结果 ----
    combined_df = (
        dataset_hits_df.unionByName(snapshot_hits_df)
        if snapshot_hits_df is not None and snapshot_hit_count > 0
        else dataset_hits_df
    )

    # ---- 4. 统计未找到的记录 ----
    found_count = dataset_hit_count + snapshot_hit_count
    unfound_count = total_candidate_count - found_count
    if unfound_count > 0:
        print(f"[WARN] {unfound_count} candidate(s) not found in dataset or snapshot, will be skipped.")
        # ponytail: 未找到的记录留存在 retry_config_downstream 中不会被清理，
        # 下次重试时仍会处理。如需清理这些记录，在此处加 DELETE 逻辑。

    # ---- 5. 组装 topic + 生成 Kafka DataFrame ----
    topic_mapping_df = _build_topic_mapping_df(
        retry_type=retry_type,
        market_codes_df=candidate_df.select("market_code").distinct(),
    )

    kafka_df = (
        combined_df.alias("d")
        .join(
            topic_mapping_df.alias("t"),
            F.col("d.market_code") == F.col("t.market_code"),
            "left",
        )
        .select(
            F.col("d.market_code").alias("market_code"),
            F.col("d.mdm_key").alias("mdm_key"),
            F.col("t.kafka_topic").alias("kafka_topic"),
            F.col("d.mdm_key").alias("kafka_key"),
            F.col("d.json_value").alias("kafka_value"),
        )
        .filter(F.col("kafka_topic").isNotNull() & (F.length(F.col("kafka_topic")) > 0))
        .filter(F.col("kafka_value").isNotNull() & (F.length(F.col("kafka_value")) > 0))
        .dropDuplicates(["market_code", "mdm_key", "kafka_topic", "kafka_key", "kafka_value"])
    )
    return kafka_df

def send_retry_messages_to_kafka(kafka_df):
    """
    将重试消息发送到 Kafka。

    参数:
        kafka_df (DataFrame): 包含 kafka_topic, kafka_key, kafka_value。
    """
    (
        kafka_df
        .select(
            F.col("kafka_key").alias("key"),
            F.col("kafka_value").alias("value"),
            F.col("kafka_topic").alias("topic"),
        )
        .write.format("kafka")
        .option("kafka.bootstrap.servers", kafka_brokers)
        .option("kafka.request.timeout.ms", "15000")
        .option("kafka.max.block.ms", "20000")
        .option("kafka.delivery.timeout.ms", "30000")
        .option("kafka.retries", "0")
        .mode("append")
        .save()
    )

def clear_retry_config_downstream_table(kafka_df):
    """
    仅清理本次实际发送成功的配置行，未发送的记录保留在配置表中。

    使用 Delta MERGE 按 (MarketCode, MDMKey) 精确匹配删除，不再全量 truncate。
    """
    from delta.tables import DeltaTable

    sent_keys_df = kafka_df.select("market_code", "mdm_key").distinct()
    sent_count = sent_keys_df.count()

    if sent_count == 0:
        print("[INFO] No sent records, config table not modified.")
        return

    config_df = spark.table(retry_config_downstream_table)
    market_col = _find_column_name(config_df, "MarketCode")
    mdm_key_col = _find_column_name(config_df, "MDMKey")

    delta_table = DeltaTable.forName(spark, retry_config_downstream_table)

    delta_table.alias("t").merge(
        sent_keys_df.alias("s"),
        f"UPPER(TRIM(t.`{market_col}`)) = s.market_code AND TRIM(t.`{mdm_key_col}`) = s.mdm_key"
    ).whenMatchedDelete().execute()

    print(f"[INFO] Deleted {sent_count} sent record(s) from retry config table: {retry_config_downstream_table}")

def retry_for_downstream(retry_type):
    """
    下游重试主流程：
      1. 校验 retry_type；
      2. 全量读取 retry_config_downstream 获取候选键；
      3. 从对应 dataset 表回查 JSON，未命中降级到 snapshot delta table；
      4. 组装 topic + 发送 Kafka；
      5. 重发成功后仅删除已发送的配置行；
      6. 输出统计信息（dataset hits / snapshot hits / unfound）。
    """
    if retry_type not in RETRY_TYPE_CONFIG:
        raise ValueError(f"Unsupported retry_type: {retry_type}")

    candidate_df = None
    kafka_df = None
    try:
        candidate_df = find_retry_candidate_df(retry_type=retry_type).cache()
        candidate_count = candidate_df.count()

        print(f"[INFO] retry_type: {retry_type}")
        print(f"[INFO] retry candidate count (from retry_config_downstream): {candidate_count}")

        if candidate_count == 0:
            if retry_type == "CBRDrjart":
                print("[INFO] no KOR candidate found for CBRDrjart in retry_config_downstream.")
            print("[INFO] no candidate found, skip sending.")
            return

        # 识别无 topic 映射的市场：直接失败，避免部分数据未发送却误清空配置表。
        market_df = candidate_df.select("market_code").distinct()
        topic_map_df = _build_topic_mapping_df(retry_type, market_df)
        missing_topic_market_df = topic_map_df.filter(F.col("kafka_topic").isNull())
        missing_topic_markets = [r["market_code"] for r in missing_topic_market_df.collect()]
        if len(missing_topic_markets) > 0:
            raise ValueError(
                "Missing topic mapping for markets: "
                + ",".join(sorted(missing_topic_markets))
                + ". Please update MARKET_TOPIC_SUFFIX / RETRY_TYPE_MARKET_TOPIC_OVERRIDE."
            )

        kafka_df = build_kafka_retry_df(candidate_df, retry_type).cache()
        kafka_count = kafka_df.count()

        print(f"[INFO] kafka retry message count: {kafka_count}")

        if kafka_count == 0:
            raise ValueError("No kafka message generated. Retry config table will NOT be modified.")

        send_retry_messages_to_kafka(kafka_df)

        print("[INFO] downstream retry messages sent successfully.")

        # 仅删除本次实际发送成功的配置行，未发送的保留。
        clear_retry_config_downstream_table(kafka_df)

    finally:
        if candidate_df is not None:
            candidate_df.unpersist()
        if kafka_df is not None:
            kafka_df.unpersist()

# main
retry_for_downstream(retry_type=retry_type)